# Notebook 02 — IPL Dataset Audit

## Objective
Inspect and validate the original IPL datasets before preprocessing.

## Audit scope
1. Dataset discovery
2. File size and dimensions
3. Column names and data types
4. Missing values
5. Duplicate records
6. Unique values and categorical distributions
7. Numerical summaries
8. Primary-key and relationship checks
9. Potential data leakage investigation

## Important
Original raw datasets must remain unchanged.

In [3]:

from pathlib import Path
import sys
import json
import pandas as pd

def find_project_root(start_path=None):
    current = (start_path or Path.cwd()).resolve()

    for candidate in (current, *current.parents):
        if (
            (candidate / "notebooks").is_dir()
            and (candidate / "src").is_dir()
            and (candidate / "data").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "Project root not found. Open this notebook "
        "from the IPL Analytics V2 project."
    )

PROJECT_ROOT = find_project_root()

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "ipl_json"

print("Project root:", PROJECT_ROOT)
print("Raw data directory:", RAW_DATA_DIR)
print("Python executable:", sys.executable)

assert PROJECT_ROOT.exists()
assert RAW_DATA_DIR.exists(), (
    f"JSON directory not found: {RAW_DATA_DIR}"
)

print("\nPROJECT PATH CONFIGURATION: PASS")


Project root: D:\Git\Personal\ipl-player-performance-v2
Raw data directory: D:\Git\Personal\ipl-player-performance-v2\data\raw\ipl_json
Python executable: c:\Users\Vinod\anaconda3\envs\ipl-analytics-v2\python.exe

PROJECT PATH CONFIGURATION: PASS


In [5]:
from pathlib import Path
import json
import pandas as pd

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "ipl_json"

json_files = sorted(RAW_DATA_DIR.glob("*.json"))

print("Raw data directory:", RAW_DATA_DIR)
print("JSON files discovered:", len(json_files))

for file in json_files[:10]:
    print(f"{file.name} | {file.stat().st_size / 1024:.2f} KB")

if not json_files:
    raise FileNotFoundError("No IPL JSON files found.")

Raw data directory: D:\Git\Personal\ipl-player-performance-v2\data\raw\ipl_json
JSON files discovered: 1243
1082591.json | 86.32 KB
1082592.json | 84.48 KB
1082593.json | 73.44 KB
1082594.json | 84.33 KB
1082595.json | 85.54 KB
1082596.json | 73.98 KB
1082597.json | 87.71 KB
1082598.json | 72.65 KB
1082599.json | 79.02 KB
1082600.json | 82.71 KB


In [6]:
datasets = {}
failed_files = []

for file in json_files:
    try:
        with file.open("r", encoding="utf-8") as f:
            match_data = json.load(f)

        datasets[file.stem] = match_data

    except Exception as exc:
        failed_files.append((file.name, str(exc)))

print("Successfully loaded:", len(datasets))
print("Failed files:", len(failed_files))

if datasets:
    sample_match = next(iter(datasets.values()))

    print("\nTop-level keys:", list(sample_match.keys()))
    print("\nMetadata keys:", list(sample_match["meta"].keys()))
    print("\nMatch information keys:", list(sample_match["info"].keys()))
    print("\nNumber of innings:", len(sample_match["innings"]))

    print("\nFailed file details:", failed_files[:10])

Successfully loaded: 1243
Failed files: 0

Top-level keys: ['meta', 'info', 'innings']

Metadata keys: ['data_version', 'created', 'revision']

Match information keys: ['balls_per_over', 'city', 'dates', 'event', 'gender', 'match_type', 'officials', 'outcome', 'overs', 'player_of_match', 'players', 'registry', 'season', 'team_type', 'teams', 'toss', 'venue']

Number of innings: 2

Failed file details: []


In [8]:
sample_match = next(iter(datasets.values()))

print("Season:", sample_match["info"].get("season"))
print("Teams:", sample_match["info"].get("teams"))
print("Venue:", sample_match["info"].get("venue"))
print("Dates:", sample_match["info"].get("dates"))

for innings_number, innings in enumerate(
    sample_match["innings"], start=1
):
    print(f"\nInnings {innings_number}")
    print("Batting team:", innings["team"])
    print("Overs:", len(innings["overs"]))

    first_over = innings["overs"][0]

    if first_over["deliveries"]:
        print("First delivery:")
        display(pd.json_normalize(first_over["deliveries"][0]))

Season: 2017
Teams: ['Sunrisers Hyderabad', 'Royal Challengers Bangalore']
Venue: Rajiv Gandhi International Stadium, Uppal
Dates: ['2017-04-05']

Innings 1
Batting team: Sunrisers Hyderabad
Overs: 20
First delivery:


,actual_delivery,batter,bowler,non_striker,runs.batter,runs.extras,runs.total
0,0.1,DA Warner,TS Mills,S Dhawan,0,0,0



Innings 2
Batting team: Royal Challengers Bangalore
Overs: 20
First delivery:


,actual_delivery,batter,bowler,non_striker,runs.batter,runs.extras,runs.total
0,0.1,CH Gayle,A Nehra,Mandeep Singh,1,0,1


In [9]:
first_delivery = sample_match["innings"][0]["overs"][0]["deliveries"][0]

print("First delivery JSON:")
print(json.dumps(first_delivery, indent=4))

print("\nDelivery keys:")
print(list(first_delivery.keys()))

display(pd.json_normalize([first_delivery]))

First delivery JSON:
{
    "actual_delivery": "0.1",
    "batter": "DA Warner",
    "bowler": "TS Mills",
    "non_striker": "S Dhawan",
    "runs": {
        "batter": 0,
        "extras": 0,
        "total": 0
    }
}

Delivery keys:
['actual_delivery', 'batter', 'bowler', 'non_striker', 'runs']


,actual_delivery,batter,bowler,non_striker,runs.batter,runs.extras,runs.total
0,0.1,DA Warner,TS Mills,S Dhawan,0,0,0


In [11]:
total_innings = 0
total_overs = 0
total_deliveries = 0

for match_id, match in datasets.items():

    for innings in match.get("innings", []):

        total_innings += 1
        total_overs += len(innings.get("overs", []))

        for over in innings.get("overs", []):
            total_deliveries += len(over.get("deliveries", []))

print("Total matches loaded:", len(datasets))
print("Total innings:", total_innings)
print("Total overs:", total_overs)
print("Total deliveries:", total_deliveries)

Total matches loaded: 1243
Total innings: 2514
Total overs: 47790
Total deliveries: 295732


In [12]:
match_records = []

for match_id, match in datasets.items():

    info = match.get("info", {})

    match_records.append({
        "match_id": match_id,
        "season": info.get("season"),
        "date": info.get("dates", [None])[0],
        "venue": info.get("venue"),
        "city": info.get("city"),
        "team1": info.get("teams", [None, None])[0],
        "team2": info.get("teams", [None, None])[1],
        "toss_winner": info.get("toss", {}).get("winner"),
        "winner": info.get("outcome", {}).get("winner"),
    })

matches_df = pd.DataFrame(match_records)

display(matches_df.head(10))

print("\nDataset dimensions:", matches_df.shape)
print("\nMissing values:")
display(matches_df.isnull().sum())

,match_id,season,date,venue,city,team1,team2,toss_winner,winner
0,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,Royal Challengers Bangalore,Royal Challengers Bangalore,Sunrisers Hyderabad
1,1082592,2017,2017-04-06,Maharashtra Cricket Association Stadium,Pune,Rising Pune Supergiant,Mumbai Indians,Rising Pune Supergiant,Rising Pune Supergiant
2,1082593,2017,2017-04-07,Saurashtra Cricket Association Stadium,Rajkot,Gujarat Lions,Kolkata Knight Riders,Kolkata Knight Riders,Kolkata Knight Riders
3,1082594,2017,2017-04-08,Holkar Cricket Stadium,Indore,Kings XI Punjab,Rising Pune Supergiant,Kings XI Punjab,Kings XI Punjab
4,1082595,2017,2017-04-08,M.Chinnaswamy Stadium,Bengaluru,Royal Challengers Bangalore,Delhi Daredevils,Royal Challengers Bangalore,Royal Challengers Bangalore
5,1082596,2017,2017-04-09,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,Gujarat Lions,Sunrisers Hyderabad,Sunrisers Hyderabad
6,1082597,2017,2017-04-09,Wankhede Stadium,Mumbai,Mumbai Indians,Kolkata Knight Riders,Mumbai Indians,Mumbai Indians
7,1082598,2017,2017-04-10,Holkar Cricket Stadium,Indore,Kings XI Punjab,Royal Challengers Bangalore,Royal Challengers Bangalore,Kings XI Punjab
8,1082599,2017,2017-04-11,Maharashtra Cricket Association Stadium,Pune,Rising Pune Supergiant,Delhi Daredevils,Rising Pune Supergiant,Delhi Daredevils
9,1082600,2017,2017-04-12,Wankhede Stadium,Mumbai,Mumbai Indians,Sunrisers Hyderabad,Mumbai Indians,Mumbai Indians



Dataset dimensions: (1243, 9)

Missing values:


match_id        0
season          0
date            0
venue           0
city           51
team1           0
team2           0
toss_winner     0
winner         25
dtype: int64

In [13]:
metadata_columns = [
    "season",
    "date",
    "venue",
    "city",
    "team1",
    "team2",
    "toss_winner",
    "winner",
]

missing_summary = pd.DataFrame({
    "missing_count": matches_df[metadata_columns].isnull().sum(),
    "missing_percentage": (
        matches_df[metadata_columns].isnull().mean() * 100
    ).round(2)
})

display(missing_summary)

,missing_count,missing_percentage
season,0,0.00
date,0,0.00
venue,0,0.00
city,51,4.10
team1,0,0.00
team2,0,0.00
toss_winner,0,0.00
winner,25,2.01


In [14]:
print("Duplicate match IDs:", matches_df["match_id"].duplicated().sum())

duplicate_match_details = matches_df[
    matches_df.duplicated(
        subset=["date", "team1", "team2", "venue"],
        keep=False
    )
].sort_values(["date", "team1", "team2"])

display(duplicate_match_details.head(20))

Duplicate match IDs: 0


,match_id,season,date,venue,city,team1,team2,toss_winner,winner


In [15]:
delivery_records = []

for match_id, match in datasets.items():

    for innings_number, innings in enumerate(
        match.get("innings", []), start=1
    ):

        for over_data in innings.get("overs", []):

            for ball_number, delivery in enumerate(
                over_data.get("deliveries", []), start=1
            ):

                delivery_records.append({
                    "match_id": match_id,
                    "innings": innings_number,
                    "batting_team": innings.get("team"),
                    "over": over_data.get("over"),
                    "delivery_number": ball_number,
                    "batter": delivery.get("batter"),
                    "bowler": delivery.get("bowler"),
                    "non_striker": delivery.get("non_striker"),
                    "total_runs": delivery.get("runs", {}).get("total"),
                    "batter_runs": delivery.get("runs", {}).get("batter"),
                    "extras_runs": delivery.get("runs", {}).get("extras"),
                    "extras": delivery.get("extras", {}),
                    "wickets": delivery.get("wickets", []),
                })

deliveries_df = pd.DataFrame(delivery_records)

print("Delivery dataset shape:", deliveries_df.shape)

display(deliveries_df.head())
display(deliveries_df.isnull().sum().to_frame("Missing values"))

Delivery dataset shape: (295732, 13)


,match_id,innings,batting_team,over,delivery_number,batter,bowler,non_striker,total_runs,batter_runs,extras_runs,extras,wickets
0,1082591,1,Sunrisers Hyderabad,0,1,DA Warner,TS Mills,S Dhawan,0,0,0,{},[]
1,1082591,1,Sunrisers Hyderabad,0,2,DA Warner,TS Mills,S Dhawan,0,0,0,{},[]
2,1082591,1,Sunrisers Hyderabad,0,3,DA Warner,TS Mills,S Dhawan,4,4,0,{},[]
3,1082591,1,Sunrisers Hyderabad,0,4,DA Warner,TS Mills,S Dhawan,0,0,0,{},[]
4,1082591,1,Sunrisers Hyderabad,0,5,DA Warner,TS Mills,S Dhawan,2,0,2,{'wides': 2},[]


,Missing values
match_id,0
innings,0
batting_team,0
over,0
delivery_number,0
batter,0
bowler,0
non_striker,0
total_runs,0
batter_runs,0


In [16]:
delivery_missing = pd.DataFrame({
    "missing_count": deliveries_df.isnull().sum(),
    "missing_percentage": (
        deliveries_df.isnull().mean() * 100
    ).round(2)
})

display(delivery_missing)

print(
    "Total missing cells:",
    deliveries_df.isnull().sum().sum()
)

,missing_count,missing_percentage
match_id,0,0.0
innings,0,0.0
batting_team,0,0.0
over,0,0.0
delivery_number,0,0.0
batter,0,0.0
bowler,0,0.0
non_striker,0,0.0
total_runs,0,0.0
batter_runs,0,0.0


Total missing cells: 0


In [17]:
delivery_key = [
    "match_id",
    "innings",
    "over",
    "delivery_number"
]

duplicate_deliveries = deliveries_df[
    deliveries_df.duplicated(
        subset=delivery_key,
        keep=False
    )
].sort_values(delivery_key)

print("Duplicate delivery records:", len(duplicate_deliveries))

display(duplicate_deliveries.head(20))

Duplicate delivery records: 0


,match_id,innings,batting_team,over,delivery_number,batter,bowler,non_striker,total_runs,batter_runs,extras_runs,extras,wickets


In [18]:
run_check = (
    deliveries_df["total_runs"]
    ==
    deliveries_df["batter_runs"] + deliveries_df["extras_runs"]
)

print("Total deliveries:", len(deliveries_df))
print("Valid run calculations:", run_check.sum())
print("Invalid run calculations:", (~run_check).sum())

invalid_run_records = deliveries_df[~run_check]

display(invalid_run_records.head(20))

Total deliveries: 295732
Valid run calculations: 295732
Invalid run calculations: 0


,match_id,innings,batting_team,over,delivery_number,batter,bowler,non_striker,total_runs,batter_runs,extras_runs,extras,wickets


In [19]:
missing_winner_matches = matches_df[
    matches_df["winner"].isnull()
]

print("Matches without recorded winners:",
      len(missing_winner_matches))

display(
    missing_winner_matches[
        ["match_id", "season", "date", "venue",
         "team1", "team2"]
    ].head(30)
)

Matches without recorded winners: 25


,match_id,season,date,venue,team1,team2
33,1082625,2017,2017-04-29,Saurashtra Cricket Association Stadium,Gujarat Lions,Mumbai Indians
128,1175365,2019,2019-03-30,Arun Jaitley Stadium,Kolkata Knight Riders,Delhi Capitals
167,1178424,2019,2019-04-30,M.Chinnaswamy Stadium,Royal Challengers Bangalore,Rajasthan Royals
169,1178426,2019,2019-05-02,Wankhede Stadium,Mumbai Indians,Sunrisers Hyderabad
180,1216493,2020/21,2020-09-20,Dubai International Cricket Stadium,Delhi Capitals,Kings XI Punjab
199,1216512,2020/21,2020-10-18,Sheikh Zayed Stadium,Kolkata Knight Riders,Sunrisers Hyderabad
204,1216517,2020/21,2020-10-18,Dubai International Cricket Stadium,Mumbai Indians,Kings XI Punjab
234,1216547,2020/21,2020-09-28,Dubai International Cricket Stadium,Royal Challengers Bangalore,Mumbai Indians
258,1254077,2021,2021-04-25,"MA Chidambaram Stadium, Chepauk, Chennai",Delhi Capitals,Sunrisers Hyderabad
417,1359519,2023,2023-05-03,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,Lucknow Super Giants,Chennai Super Kings


In [20]:
missing_winner_ids = set(
    missing_winner_matches["match_id"].astype(str)
)

outcome_records = []

for match_id, match in datasets.items():

    if str(match_id) not in missing_winner_ids:
        continue

    info = match.get("info", {})

    outcome_records.append({
        "match_id": match_id,
        "season": info.get("season"),
        "date": info.get("dates", [None])[0],
        "teams": info.get("teams"),
        "outcome": info.get("outcome"),
        "match_type": info.get("match_type"),
        "event": info.get("event"),
    })

outcome_df = pd.DataFrame(outcome_records)

print("Matches investigated:", len(outcome_df))

display(outcome_df)

Matches investigated: 25


,match_id,season,date,teams,outcome,match_type,event
0,1082625,2017,2017-04-29,"[Gujarat Lions, Mumbai Indians]","{'eliminator': 'Mumbai Indians', 'result': 'tie'}",T20,"{'match_number': 35, 'name': 'Indian Premier L..."
1,1175365,2019,2019-03-30,"[Kolkata Knight Riders, Delhi Capitals]","{'result': 'tie', 'eliminator': 'Delhi Capitals'}",T20,"{'name': 'Indian Premier League', 'match_numbe..."
2,1178424,2019,2019-04-30,"[Royal Challengers Bangalore, Rajasthan Royals]",{'result': 'no result'},T20,"{'name': 'Indian Premier League', 'match_numbe..."
3,1178426,2019,2019-05-02,"[Mumbai Indians, Sunrisers Hyderabad]","{'result': 'tie', 'eliminator': 'Mumbai Indians'}",T20,"{'name': 'Indian Premier League', 'match_numbe..."
4,1216493,2020/21,2020-09-20,"[Delhi Capitals, Kings XI Punjab]","{'result': 'tie', 'eliminator': 'Delhi Capitals'}",T20,"{'name': 'Indian Premier League', 'match_numbe..."
5,1216512,2020/21,2020-10-18,"[Kolkata Knight Riders, Sunrisers Hyderabad]","{'result': 'tie', 'eliminator': 'Kolkata Knigh...",T20,"{'name': 'Indian Premier League', 'match_numbe..."
6,1216517,2020/21,2020-10-18,"[Mumbai Indians, Kings XI Punjab]","{'result': 'tie', 'eliminator': 'Kings XI Punj...",T20,"{'name': 'Indian Premier League', 'match_numbe..."
7,1216547,2020/21,2020-09-28,"[Royal Challengers Bangalore, Mumbai Indians]","{'result': 'tie', 'eliminator': 'Royal Challen...",T20,"{'name': 'Indian Premier League', 'match_numbe..."
8,1254077,2021,2021-04-25,"[Delhi Capitals, Sunrisers Hyderabad]","{'result': 'tie', 'eliminator': 'Delhi Capitals'}",T20,"{'name': 'Indian Premier League', 'match_numbe..."
9,1359519,2023,2023-05-03,"[Lucknow Super Giants, Chennai Super Kings]",{'result': 'no result'},T20,"{'name': 'Indian Premier League', 'match_numbe..."


In [21]:
for match_id, match in datasets.items():

    if str(match_id) not in missing_winner_ids:
        continue

    info = match.get("info", {})
    outcome = info.get("outcome", {})

    print("\nMatch ID:", match_id)
    print("Season:", info.get("season"))
    print("Teams:", info.get("teams"))
    print("Outcome:", outcome)
    print("Result:", outcome.get("result"))
    print("Method:", outcome.get("method"))
    print("Winner:", outcome.get("winner"))


Match ID: 1082625
Season: 2017
Teams: ['Gujarat Lions', 'Mumbai Indians']
Outcome: {'eliminator': 'Mumbai Indians', 'result': 'tie'}
Result: tie
Method: None
Winner: None

Match ID: 1175365
Season: 2019
Teams: ['Kolkata Knight Riders', 'Delhi Capitals']
Outcome: {'result': 'tie', 'eliminator': 'Delhi Capitals'}
Result: tie
Method: None
Winner: None

Match ID: 1178424
Season: 2019
Teams: ['Royal Challengers Bangalore', 'Rajasthan Royals']
Outcome: {'result': 'no result'}
Result: no result
Method: None
Winner: None

Match ID: 1178426
Season: 2019
Teams: ['Mumbai Indians', 'Sunrisers Hyderabad']
Outcome: {'result': 'tie', 'eliminator': 'Mumbai Indians'}
Result: tie
Method: None
Winner: None

Match ID: 1216493
Season: 2020/21
Teams: ['Delhi Capitals', 'Kings XI Punjab']
Outcome: {'result': 'tie', 'eliminator': 'Delhi Capitals'}
Result: tie
Method: None
Winner: None

Match ID: 1216512
Season: 2020/21
Teams: ['Kolkata Knight Riders', 'Sunrisers Hyderabad']
Outcome: {'result': 'tie', 'elimin

In [22]:
display(delivery_missing)
print("Total missing cells:", deliveries_df.isnull().sum().sum())

,missing_count,missing_percentage
match_id,0,0.0
innings,0,0.0
batting_team,0,0.0
over,0,0.0
delivery_number,0,0.0
batter,0,0.0
bowler,0,0.0
non_striker,0,0.0
total_runs,0,0.0
batter_runs,0,0.0


Total missing cells: 0


In [23]:
outcome_summary = []

for match_id, match in datasets.items():

    if str(match_id) not in missing_winner_ids:
        continue

    outcome = match.get("info", {}).get("outcome", {})

    outcome_summary.append({
        "match_id": match_id,
        "result": outcome.get("result", "not specified"),
        "method": outcome.get("method"),
        "eliminator": outcome.get("eliminator"),
        "bowl_out": outcome.get("bowl_out"),
        "has_winner": outcome.get("winner") is not None
    })

outcome_audit_df = pd.DataFrame(outcome_summary)

print("Outcome distribution:")
display(
    outcome_audit_df["result"]
    .value_counts(dropna=False)
    .to_frame("match_count")
)

print("\nComplete outcome summary:")
display(outcome_audit_df)

Outcome distribution:


,match_count
result,
tie,16
no result,9



Complete outcome summary:


,match_id,result,method,eliminator,bowl_out,has_winner
0,1082625,tie,None,Mumbai Indians,None,False
1,1175365,tie,None,Delhi Capitals,None,False
2,1178424,no result,None,NaN,None,False
3,1178426,tie,None,Mumbai Indians,None,False
4,1216493,tie,None,Delhi Capitals,None,False
5,1216512,tie,None,Kolkata Knight Riders,None,False
6,1216517,tie,None,Kings XI Punjab,None,False
7,1216547,tie,None,Royal Challengers Bangalore,None,False
8,1254077,tie,None,Delhi Capitals,None,False
9,1359519,no result,None,NaN,None,False


In [24]:
missing_city_matches = matches_df[
    matches_df["city"].isnull()
]

print("Matches with missing city:",
      len(missing_city_matches))

display(
    missing_city_matches[
        ["match_id", "season", "date", "venue", "team1", "team2"]
    ]
)

Matches with missing city: 51


,match_id,season,date,venue,team1,team2
180,1216493,2020/21,2020-09-20,Dubai International Cricket Stadium,Delhi Capitals,Kings XI Punjab
182,1216495,2020/21,2020-11-03,Sharjah Cricket Stadium,Mumbai Indians,Sunrisers Hyderabad
183,1216496,2020/21,2020-09-22,Sharjah Cricket Stadium,Rajasthan Royals,Chennai Super Kings
185,1216498,2020/21,2020-10-24,Dubai International Cricket Stadium,Kings XI Punjab,Sunrisers Hyderabad
187,1216500,2020/21,2020-10-09,Sharjah Cricket Stadium,Delhi Capitals,Rajasthan Royals
189,1216502,2020/21,2020-10-31,Sharjah Cricket Stadium,Royal Challengers Bangalore,Sunrisers Hyderabad
191,1216504,2020/21,2020-09-30,Dubai International Cricket Stadium,Kolkata Knight Riders,Rajasthan Royals
194,1216507,2020/21,2020-10-11,Dubai International Cricket Stadium,Sunrisers Hyderabad,Rajasthan Royals
196,1216509,2020/21,2020-10-17,Sharjah Cricket Stadium,Chennai Super Kings,Delhi Capitals
197,1216510,2020/21,2020-09-24,Dubai International Cricket Stadium,Kings XI Punjab,Royal Challengers Bangalore


In [25]:
missing_city_detail = matches_df[
    matches_df["city"].isna()
].copy()

venue_distribution = (
    missing_city_detail
    .groupby(["season", "venue"], dropna=False)
    .size()
    .reset_index(name="match_count")
    .sort_values(["season", "match_count"], ascending=[True, False])
)

print("Missing city records:", len(missing_city_detail))
print("Unique venues:", missing_city_detail["venue"].nunique())
print("Seasons:", missing_city_detail["season"].unique())

display(venue_distribution)

Missing city records: 51
Unique venues: 2
Seasons: ['2020/21' 2014]


,season,venue,match_count
0,2014,Dubai International Cricket Stadium,7
1,2014,Sharjah Cricket Stadium,6
2,2020/21,Dubai International Cricket Stadium,26
3,2020/21,Sharjah Cricket Stadium,12


In [26]:
eliminator_distribution = (
    outcome_audit_df
    .groupby(["result", "eliminator"], dropna=False)
    .size()
    .reset_index(name="match_count")
    .sort_values("result")
)

display(eliminator_distribution)

print("\nOutcome totals:")
print(outcome_audit_df["result"].value_counts(dropna=False))

,result,eliminator,match_count
0,no result,NaN,9
1,tie,Delhi Capitals,4
2,tie,Kings XI Punjab,3
3,tie,Kolkata Knight Riders,2
4,tie,Mumbai Indians,2
5,tie,Rajasthan Royals,2
6,tie,Royal Challengers Bangalore,2
7,tie,Sunrisers Hyderabad,1



Outcome totals:
result
tie          16
no result     9
Name: count, dtype: int64


In [28]:

# Create a temporary string representation for consistent sorting
season_counts = (
    missing_city_detail["season"]
    .astype("string")
    .value_counts(dropna=False)
)

display(
    season_counts
    .sort_index()
    .to_frame("match_count")
)

# Check whether all missing city records belong to 2020/21
all_2020_21 = (
    missing_city_detail["season"]
    .astype("string")
    .eq("2020/21")
    .all()
)

print(
    "All missing city records belong to 2020/21:",
    all_2020_21
)


,match_count
season,
2014,13
2020/21,38


All missing city records belong to 2020/21: False


In [29]:
# Explicit mapping based on the venues identified in the audit

venue_city_mapping = {
    "Dubai International Cricket Stadium": "Dubai",
    "Sharjah Cricket Stadium": "Sharjah",
}

# Create a separate analytical copy
matches_audited = matches_df.copy()

# Preserve original city values
matches_audited["city_original"] = matches_audited["city"]

# Identify records whose city is missing
missing_city_mask = matches_audited["city"].isna()

# Fill only missing city values with a verified venue mapping
matches_audited.loc[missing_city_mask, "city"] = (
    matches_audited.loc[missing_city_mask, "venue"]
    .map(venue_city_mapping)
)

# Track which records were filled
matches_audited["city_was_imputed"] = missing_city_mask

print("Original missing city values:",
      matches_df["city"].isna().sum())

print("Remaining missing city values:",
      matches_audited["city"].isna().sum())

print("City values filled:",
      matches_audited["city_was_imputed"].sum())

display(
    matches_audited[
        matches_audited["city_was_imputed"]
    ][["match_id", "season", "venue",
       "city_original", "city", "city_was_imputed"]]
    .head(10)
)

Original missing city values: 51
Remaining missing city values: 0
City values filled: 51


,match_id,season,venue,city_original,city,city_was_imputed
180,1216493,2020/21,Dubai International Cricket Stadium,NaN,Dubai,True
182,1216495,2020/21,Sharjah Cricket Stadium,NaN,Sharjah,True
183,1216496,2020/21,Sharjah Cricket Stadium,NaN,Sharjah,True
185,1216498,2020/21,Dubai International Cricket Stadium,NaN,Dubai,True
187,1216500,2020/21,Sharjah Cricket Stadium,NaN,Sharjah,True
189,1216502,2020/21,Sharjah Cricket Stadium,NaN,Sharjah,True
191,1216504,2020/21,Dubai International Cricket Stadium,NaN,Dubai,True
194,1216507,2020/21,Dubai International Cricket Stadium,NaN,Dubai,True
196,1216509,2020/21,Sharjah Cricket Stadium,NaN,Sharjah,True
197,1216510,2020/21,Dubai International Cricket Stadium,NaN,Dubai,True


In [30]:
unmapped_missing_cities = matches_audited[
    matches_audited["city"].isna()
]

print("Unmapped missing city records:",
      len(unmapped_missing_cities))

print("\nCity distribution after mapping:")

display(
    matches_audited["city"]
    .value_counts(dropna=False)
    .head(20)
)

assert len(unmapped_missing_cities) == 0

assert matches_audited["city_was_imputed"].sum() == 51

print("\nVENUE-TO-CITY MAPPING VALIDATION: PASS")

Unmapped missing city records: 0

City distribution after mapping:


city
Mumbai           187
Kolkata          107
Delhi            104
Chennai           98
Hyderabad         90
Jaipur            68
Bangalore         65
Chandigarh        61
Ahmedabad         53
Pune              51
Dubai             46
Bengaluru         39
Abu Dhabi         37
Lucknow           29
Sharjah           28
Dharamsala        19
Visakhapatnam     17
Durban            15
Centurion         12
Rajkot            10
Name: count, dtype: int64


VENUE-TO-CITY MAPPING VALIDATION: PASS


In [31]:
from collections import Counter

extras_counter = Counter()

for match in datasets.values():
    for innings in match.get("innings", []):
        for over_data in innings.get("overs", []):
            for delivery in over_data.get("deliveries", []):

                extras = delivery.get("extras", {})

                for extra_type, runs in extras.items():
                    extras_counter[extra_type] += 1

print("Extras categories found:\n")

for extra_type, count in extras_counter.most_common():
    print(f"{extra_type}: {count}")

print("\nTotal extras categories:", len(extras_counter))

Extras categories found:

wides: 9876
legbyes: 4406
noballs: 1226
byes: 745
penalty: 2

Total extras categories: 5


In [32]:
wicket_counter = Counter()

for match in datasets.values():
    for innings in match.get("innings", []):
        for over_data in innings.get("overs", []):
            for delivery in over_data.get("deliveries", []):

                for wicket in delivery.get("wickets", []):
                    wicket_counter[wicket.get("kind")] += 1

print("Wicket types found:\n")

for wicket_type, count in wicket_counter.most_common():
    print(f"{wicket_type}: {count}")

print("\nTotal wicket types:", len(wicket_counter))
print("Total wicket records:", sum(wicket_counter.values()))

Wicket types found:

caught: 9321
bowled: 2460
run out: 1194
lbw: 883
caught and bowled: 410
stumped: 388
hit wicket: 20
retired hurt: 19
retired out: 6
obstructing the field: 4

Total wicket types: 10
Total wicket records: 14705


In [33]:

legal_delivery_count = 0
illegal_delivery_count = 0

illegal_delivery_types = Counter()

for match in datasets.values():
    for innings in match.get("innings", []):
        for over_data in innings.get("overs", []):
            for delivery in over_data.get("deliveries", []):

                extras = delivery.get("extras", {})

                is_wide = "wides" in extras
                is_noball = "noballs" in extras

                if is_wide or is_noball:
                    illegal_delivery_count += 1

                    if is_wide:
                        illegal_delivery_types["wide"] += 1

                    if is_noball:
                        illegal_delivery_types["no-ball"] += 1
                else:
                    legal_delivery_count += 1

print("Legal deliveries:", legal_delivery_count)
print("Illegal deliveries:", illegal_delivery_count)
print("Total deliveries:", legal_delivery_count + illegal_delivery_count)

print("\nIllegal delivery breakdown:")
for delivery_type, count in illegal_delivery_types.items():
    print(f"{delivery_type}: {count}")

Legal deliveries: 284630
Illegal deliveries: 11102
Total deliveries: 295732

Illegal delivery breakdown:
wide: 9876
no-ball: 1226


In [36]:

from collections import Counter

bowler_credited_types = {
    "bowled",
    "caught",
    "caught and bowled",
    "lbw",
    "stumped",
    "hit wicket",
}

non_bowler_dismissal_types = {
    "run out",
    "retired out",
    "obstructing the field",
}

non_dismissal_records = {
    "retired hurt",
}

bowler_wickets = Counter()
other_dismissals = Counter()
non_dismissal_events = Counter()

for match in datasets.values():
    for innings in match.get("innings", []):
        for over_data in innings.get("overs", []):
            for delivery in over_data.get("deliveries", []):

                for wicket in delivery.get("wickets", []):

                    kind = wicket.get("kind")

                    if kind in bowler_credited_types:
                        bowler_wickets[kind] += 1

                    elif kind in non_bowler_dismissal_types:
                        other_dismissals[kind] += 1

                    elif kind in non_dismissal_records:
                        non_dismissal_events[kind] += 1

                    else:
                        raise ValueError(
                            f"Unclassified wicket event: {kind}"
                        )

bowler_total = sum(bowler_wickets.values())
other_total = sum(other_dismissals.values())
non_dismissal_total = sum(non_dismissal_events.values())

actual_dismissals = bowler_total + other_total
all_wicket_records = actual_dismissals + non_dismissal_total

print("Bowler-creditable dismissals:", bowler_total)
print("Other actual dismissals:", other_total)
print("Non-dismissal events:", non_dismissal_total)
print("Actual dismissals:", actual_dismissals)
print("Total JSON wicket records:", all_wicket_records)

assert bowler_total == 13482
assert other_total == 1204
assert non_dismissal_total == 19
assert actual_dismissals == 14686
assert all_wicket_records == 14705

print("\nWICKET CLASSIFICATION VALIDATION: PASS")


Bowler-creditable dismissals: 13482
Other actual dismissals: 1204
Non-dismissal events: 19
Actual dismissals: 14686
Total JSON wicket records: 14705

WICKET CLASSIFICATION VALIDATION: PASS


In [37]:
assert sum(extras_counter.values()) == (
    sum(extras_counter.values())
)

assert legal_delivery_count + illegal_delivery_count == len(deliveries_df)

assert illegal_delivery_types["wide"] == extras_counter["wides"]
assert illegal_delivery_types["no-ball"] == extras_counter["noballs"]

print("Extras categories:", dict(extras_counter))
print("Legal deliveries:", legal_delivery_count)
print("Illegal deliveries:", illegal_delivery_count)
print("Total deliveries:", len(deliveries_df))

print("\nDELIVERY RECONCILIATION: PASS")

Extras categories: {'wides': 9876, 'legbyes': 4406, 'noballs': 1226, 'byes': 745, 'penalty': 2}
Legal deliveries: 284630
Illegal deliveries: 11102
Total deliveries: 295732

DELIVERY RECONCILIATION: PASS


In [38]:

# Build a match-to-teams lookup from original JSON
match_team_lookup = {
    str(match_id): set(match.get("info", {}).get("teams", []))
    for match_id, match in datasets.items()
}

# Check match IDs
json_match_ids = set(match_team_lookup.keys())
delivery_match_ids = set(deliveries_df["match_id"].astype(str))
metadata_match_ids = set(matches_df["match_id"].astype(str))

missing_match_ids = delivery_match_ids - json_match_ids
missing_delivery_matches = json_match_ids - delivery_match_ids
missing_metadata_matches = json_match_ids - metadata_match_ids

# Check whether each delivery's batting team belongs to its match
invalid_team_rows = deliveries_df[
    ~deliveries_df.apply(
        lambda row: row["batting_team"] in
        match_team_lookup.get(str(row["match_id"]), set()),
        axis=1
    )
]

print("Matches in JSON:", len(json_match_ids))
print("Matches in metadata:", len(metadata_match_ids))
print("Matches represented in deliveries:", len(delivery_match_ids))

print("\nDelivery match IDs without JSON match:", len(missing_match_ids))
print("JSON matches without deliveries:", len(missing_delivery_matches))
print("JSON matches without metadata:", len(missing_metadata_matches))
print("Deliveries with invalid batting teams:", len(invalid_team_rows))

assert not missing_match_ids
assert not missing_delivery_matches
assert not missing_metadata_matches
assert invalid_team_rows.empty

print("\nMATCH-DELIVERY REFERENTIAL INTEGRITY: PASS")


Matches in JSON: 1243
Matches in metadata: 1243
Matches represented in deliveries: 1243

Delivery match IDs without JSON match: 0
JSON matches without deliveries: 0
JSON matches without metadata: 0
Deliveries with invalid batting teams: 0

MATCH-DELIVERY REFERENTIAL INTEGRITY: PASS
